# Indie Games on Steam: AI Content Disclosures

This is the second part of this project. Since January 2024, Steam asks developers to disclore AI-generated content, shown on the page in a section called "AI Generated Content Disclosure".

This information is not in the dataset, so it is collected from each game's store page.

**How the disclosure is detected:** the section is inside an element with id `game_area_content_descriptors`. Steam uses the same id for the "Mature Content Description" section, so the section's heading is also checked. Pages are requested in English so the heading text is always the same.

In [2]:
import requests
from bs4 import BeautifulSoup

COOKIES = {
    "birthtime": "568022401",
    "lastagecheckage": "1-0-1988",
    "wants_mature_content": "1",
}
HEADERS = {"User-Agent": "Mozilla/5.0 (student research project)"}

def get_ai_disclosure(appid):
    url = f"https://store.steampowered.com/app/{appid}"
    response = requests.get(url, params={"l": "english"}, cookies=COOKIES, headers=HEADERS, timeout=20)

    if response.status_code != 200:
        return {"AppID": appid, "status": f"http_{response.status_code}", "has_ai": None, "ai_text": None}
    
    soup = BeautifulSoup(response.text, "html.parser")

    if soup.find(id="appHubAppName") is None:
        return {"AppID": appid, "status": "no_store_page", "has_ai": None, "ai_text": None}

    for section in soup.find_all(id="game_area_content_descriptors"):
        heading = section.find("h2")
        if heading and "AI Generated Content" in heading.get_text():
            return {"AppID": appid, "status": "ok", "has_ai": True, "ai_text": section.get_text(" ", strip=True)}
    
    return {"AppID": appid, "status": "ok", "has_ai": False, "ai_text": None}

print(get_ai_disclosure(2456740)) # Example usage with inZOI (should have AI)
print(get_ai_disclosure(367520)) #Example usage with Hollow Knight (should not)

{'AppID': 2456740, 'status': 'ok', 'has_ai': True, 'ai_text': 'AI Generated Content Disclosure The developers describe how their game uses AI Generated Content like this: Players can generate unique textures for character outfits and various items based on text input. They can also create 3D objects from 2D image input, which can be used as interior decorations or character accessories, and add distinctive motions to their Zoi using video input. Additionally, character actions and thoughts can be influenced with text using SLM technology, enabling more engaging and intuitive interactions. AI-based tools are utilized as auxiliary support during the content creation process and text localizations. All content generated through this process was thoroughly reviewed and manually refined by the development team before applied to the final product.'}
{'AppID': 367520, 'status': 'ok', 'has_ai': False, 'ai_text': None}


**Testing edge cases:**

- A mature game without AI (has a "Mature Content" section, that uses the same ID as the "AI Generated Content") that should return `False`, also confirming that the age check is passed.

- An AppID that doesn't exist, that should return `no_store_page`, not `False`. (to distinguish not knowing from not having)

In [3]:
print(get_ai_disclosure(1091500))   # Cyberpunk 2077: mature content, no AI disclosure expected
print(get_ai_disclosure(1))         # AppID that doesn't exist

{'AppID': 1091500, 'status': 'ok', 'has_ai': False, 'ai_text': None}
{'AppID': 1, 'status': 'no_store_page', 'has_ai': None, 'ai_text': None}


## Collecting disclosures for a sample of games

**Population:** indie games released in 2024 and 2025. The disclosure was introduced in January 2024, and Valve changed the disclosure rules in January 2026, so this window keeps the rules consistent.

**Sample:** 2000 games chosen at random from this population. This is done because collecting all ~28000 pages would take hours, and a sample gives representative results in a fraction of the time. A fixed random seed makes the sample reproductible.

**How the data is collected:**
- There is a pause between requests, as a way to avoid overloading Steam's servers.
- Each result is saved to a file immediately, so if collection stops halfway it resumes from where it stopped instead of starting over.
- Errors are recorded rather than ignored, so games that couldn't be checked out are not counted as "No AI".

In [4]:
import pandas as pd

df = pd.read_csv("data/games_clean.csv")
population = df[df["Is indie"] & df["Year"].isin([2024,2025])]
print("Population:", len(population))

sample = population.sample(n=2000, random_state=42)
print("Sample:", len(sample))

Population: 28073
Sample: 2000


In [9]:
import os
import time

OUTPUT = "data/ai_disclosures.csv"

def collect(appids, limit=None):
    done = set(pd.read_csv(OUTPUT)["AppID"]) if os.path.exists(OUTPUT) else set()
    todo = [appid for appid in appids if appid not in done]
    if limit:
        todo = todo[:limit]
    print(f"Already done: {len(done)} | To do now: {len(todo)}")

    for i, appid in enumerate(todo, start=1):
        try:
            result = get_ai_disclosure(appid)
        except requests.RequestException as e:
            result = {"AppID": appid, "status": f"error: {type(e).__name__}",
                      "has_ai": None, "ai_text": None}

        pd.DataFrame([result]).to_csv(OUTPUT, mode="a", index=False,
                                      header=not os.path.exists(OUTPUT))

        if result["status"] == "http_429":
            print("Too many requests — waiting 60 seconds")
            time.sleep(60)

        if i % 50 == 0 or i == len(todo):
            print(f"{i}/{len(todo)} done")

        time.sleep(1.5)

In [10]:
# Test run
collect(sample["AppID"], limit=20)

Already done: 0 | To do now: 20
20/20 done


After confirming the logic works for a sample with size = 20, we pass onto the bigger sample.

In [11]:
# Full run
collect(sample["AppID"])

Already done: 20 | To do now: 1980
50/1980 done
100/1980 done
150/1980 done
200/1980 done
250/1980 done
300/1980 done
350/1980 done
400/1980 done
450/1980 done
500/1980 done
550/1980 done
600/1980 done
650/1980 done
700/1980 done
750/1980 done
800/1980 done
850/1980 done
900/1980 done
950/1980 done
1000/1980 done
1050/1980 done
1100/1980 done
1150/1980 done
1200/1980 done
1250/1980 done
1300/1980 done
1350/1980 done
1400/1980 done
1450/1980 done
1500/1980 done
1550/1980 done
1600/1980 done
1650/1980 done
1700/1980 done
1750/1980 done
1800/1980 done
1850/1980 done
1900/1980 done
1950/1980 done
1980/1980 done


In [12]:
results = pd.read_csv(OUTPUT)
print("Total:", len(results))
print(results["status"].value_counts())
print(results["has_ai"].value_counts(dropna=False))

Total: 2000
status
ok                        1854
no_store_page              142
error: ConnectionError       4
Name: count, dtype: int64
has_ai
False    1543
True      311
NaN       146
Name: count, dtype: int64


**Collection results:** all 2,000 games were processed.
- 1,854 pages were checked successfully, and 311 of them disclose AI-generated content.
- 142 pages (7.1%) could not be opened (`no_store_page`). This is investigated below, since excluding a specific group of games could bias the results.
- 4 requests failed due to connection errors and are retried in the next cell.

In [13]:
results = pd.read_csv(OUTPUT)
results = results[~results["status"].str.startswith("error")]
results.to_csv(OUTPUT, index=False)
print("Rows kept:", len(results))

collect(sample["AppID"])

Rows kept: 1996
Already done: 1996 | To do now: 4
4/4 done


In [14]:
missing_ids = results.loc[results["status"] == "no_store_page", "AppID"]
missing = sample[sample["AppID"].isin(missing_ids)]

def share_with(games, label):
    return games["Tags"].fillna("").str.contains(label).mean() * 100

for label in ["Sexual Content", "Nudity"]:
    print(f"{label}: {share_with(missing, label):.1f}% of missing pages "
          f"vs {share_with(sample, label):.1f}% of the whole sample")

missing[["AppID", "Name", "Genres", "Total reviews"]].head(10)

Sexual Content: 0.7% of missing pages vs 0.9% of the whole sample
Nudity: 0.7% of missing pages vs 0.4% of the whole sample


,AppID,Name,Genres,Total reviews
17620,4165160,A Lucky Hunt With : Ione,"Casual,Indie",0
83550,4024870,Hentai Clicker: Valentine is streaming,"Casual,Indie",0
41426,3615790,A Sexy Tour With : Yeona,"Casual,Indie",0
23434,3857900,Hentai Night Club,"Casual,Indie",0
90151,2835540,Kawaii Puzzle: Girl Adventure 🌸💖,"Adventure,Casual,Indie",0
35569,3719000,Dominated by Sadistic Rich Bully Trio,"Indie,Simulation",0
72024,2873130,Hentai Tales: The Neko Returns,"Casual,Indie,Strategy",0
98660,3530770,Monster City: A Monster College Story,"Adventure,Indie,Early Access",0
91256,2703920,Outbreak Zombie Plague,"Action,Adventure,Indie,RPG,Strategy",0
103284,3862660,Queen of the fallen land,"Action,Indie",0


In [15]:
print("Missing tags, missing pages:", missing["Tags"].isna().mean() * 100)
print("Missing tags, whole sample: ", sample["Tags"].isna().mean() * 100)

Missing tags, missing pages: 98.59154929577466
Missing tags, whole sample:  53.75


**Findings:**
- Most of these games have zero reviews, and many have explicitly adult titles. This suggests they are **adult-only games, which Steam only shows to signed-in users**, so the scraper was redirected away from their pages.
- A first check based on tags found no difference, but that check was not valid: 98.6% of these games have no tags at all (vs 53.8% in the whole sample), because games with no players often have none.
- Collecting these pages would require using a personal Steam login cookie in the code, which is a security risk, so they are excluded.

In [17]:
results = pd.read_csv(OUTPUT)
print("Total:", len(results))
print(results["status"].value_counts())
print(results["has_ai"].value_counts(dropna=False))

Total: 2000
status
ok               1858
no_store_page     142
Name: count, dtype: int64
has_ai
False    1547
True      311
NaN       142
Name: count, dtype: int64


In [18]:
checked = results[results["status"] == "ok"]
n = len(checked)
n_ai = checked["has_ai"].sum()
share = n_ai / n

margin = 1.96 * (share * (1 - share) / n) ** 0.5
print(f"AI share (checked pages): {share:.1%} ± {margin:.1%}  ({n_ai} of {n})")

n_missing = (results["status"] == "no_store_page").sum()
lower = n_ai / len(results)
upper = (n_ai + n_missing) / len(results)
print(f"Bounds including unreachable pages: {lower:.1%} to {upper:.1%}")

AI share (checked pages): 16.7% ± 1.7%  (311 of 1858)
Bounds including unreachable pages: 15.6% to 22.7%


**Result:**
- Among the 1,858 games whose store page could be checked, **16.7% disclose AI-generated content** (margin of error about ±1.7 percentage points, 95% confidence).
- Including the 142 adult-only games that could not be checked, the overall share lies between **15.6%** (if none of them disclosed AI) and **22.7%** (if all did). Since adult-only games likely use AI-generated art more often than average, the true share is probably somewhat above 16.7%.